# 🛡️ Mini SOC — GRPO Training Notebook

Train an RL agent to operate a Security Operations Center using **Group Relative Policy Optimization (GRPO)**.

This notebook:
1. Installs dependencies (TRL, PEFT)
2. Connects to the Mini SOC environment (HF Space or local)
3. Runs a random baseline to establish "before" scores
4. Runs 200-step GRPO training
5. Evaluates the trained model and generates reward curve
6. Produces a before/after comparison chart

> **GPU Required:** T4 or better. Runtime → Change runtime type → GPU → T4

> **HF Space:** https://huggingface.co/spaces/riteshp30/mini-soc

In [ ]:
# Install core training dependencies
!pip install -q \
    trl>=0.15.0 \
    peft>=0.14.0 \
    transformers>=4.48.0 \
    datasets \
    httpx \
    wandb \
    torch \
    matplotlib \
    fastapi \
    uvicorn \
    pydantic

# Clone the Mini SOC repository
!git clone https://github.com/riteshthekid/mini-soc.git 2>/dev/null || echo 'Repo already cloned'
%cd mini-soc

# Verify GPU
import torch
print(f'\n✅ Dependencies installed')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2: Connect to Environment

**Two options:**
- **Option A (Recommended):** Use the live HF Space — no setup needed
- **Option B:** Start a local server in Colab — fully self-contained

In [ ]:
import os
import time
import httpx

# ============================================
# CHOOSE ONE: 'remote' or 'local'
# ============================================
MODE = 'remote'   # 'remote' = HF Space, 'local' = local server

if MODE == 'remote':
    SOC_ENV_URL = 'https://riteshp30-mini-soc.hf.space'
    print(f'🌐 Using remote HF Space: {SOC_ENV_URL}')

    # Wake the Space if it's sleeping (free tier sleeps after 15 min)
    for attempt in range(5):
        try:
            r = httpx.get(f'{SOC_ENV_URL}/health', timeout=60)
            if r.status_code == 200:
                print(f'✅ Space is awake: {r.json()}')
                break
        except Exception as e:
            print(f'   Waiting for Space to wake... ({attempt+1}/5): {e}')
            time.sleep(15)
    else:
        print('❌ Space did not wake up. Switch to MODE="local"')
        raise RuntimeError('HF Space unreachable')

else:
    SOC_ENV_URL = 'http://localhost:8000'
    print(f'💻 Starting local server...')

    import threading
    def start_server():
        import uvicorn
        from server.app import app
        uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

    server_thread = threading.Thread(target=start_server, daemon=True)
    server_thread.start()
    time.sleep(3)

    r = httpx.get(f'{SOC_ENV_URL}/health', timeout=10)
    print(f'✅ Local server healthy: {r.json()}')

# Set environment variable for all training modules
os.environ['SOC_ENV_URL'] = SOC_ENV_URL
os.environ['MODEL_NAME'] = 'Qwen/Qwen2.5-1.5B-Instruct'

# Optional: Set your API keys
# os.environ['WANDB_API_KEY'] = 'your-wandb-key'
# os.environ['HF_TOKEN'] = 'your-hf-token'

# Quick sanity check — reset all 3 tasks
print('\n📋 Testing all 3 tasks:')
for task in ['alert_triage', 'incident_investigation', 'threat_response']:
    r = httpx.post(f'{SOC_ENV_URL}/reset', json={'task_id': task}, timeout=30)
    obs = r.json()['observation']
    print(f'   {task}: alerts={len(obs["alert_queue"])}, logs={len(obs["available_logs"])}')

print('\n✅ Environment ready for training!')

## Cell 3: Run Random Baseline ("Before" Evidence)

⚠️ **Screenshot this output!** This is your "before" evidence for judges.

In [ ]:
import httpx
import random

TASK_IDS = ['alert_triage', 'incident_investigation', 'threat_response']


def run_random_agent(task_id, num_episodes=3):
    """Run a random agent and return average final score."""
    scores = []
    actions = ['query_logs', 'classify_alert', 'isolate_asset', 'block_ip', 'request_info']

    for ep in range(num_episodes):
        httpx.post(f'{SOC_ENV_URL}/reset', json={'task_id': task_id}, timeout=30)
        done = False
        final_score = 0.0
        steps = 0

        while not done and steps < 20:
            action = random.choice(actions)
            params = {}
            if action == 'query_logs':
                params = {'log_source': random.choice(['auth', 'firewall', 'dns', 'process', 'network'])}
            elif action == 'classify_alert':
                params = {
                    'alert_id': f'ALT-{random.randint(1, 33):03d}',
                    'classification': random.choice(['benign', 'suspicious', 'critical']),
                    'priority': random.choice(['P1', 'P2', 'P3', 'P4']),
                }
            elif action == 'isolate_asset':
                params = {'hostname': random.choice(['WEB-SERVER-01', 'DC-01', 'WS-HR-03'])}
            elif action == 'block_ip':
                params = {'ip_address': f'{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}'}

            try:
                step_resp = httpx.post(
                    f'{SOC_ENV_URL}/step',
                    json={'action_type': action, 'parameters': params},
                    timeout=30,
                )
                result = step_resp.json()
                done = result.get('done', False)
                if done:
                    final_score = result.get('info', {}).get('final_score', 0.0)
                steps += 1
            except Exception:
                break

        scores.append(final_score)

    return sum(scores) / len(scores) if scores else 0.0


# Run random agent baseline
print('Running random agent baseline (3 episodes each)...')
print()
baseline_scores = {}
for task in TASK_IDS:
    score = run_random_agent(task)
    baseline_scores[task] = round(score, 4)
    print(f'   {task:<30} {score:.4f}')

avg = sum(baseline_scores.values()) / len(baseline_scores)
print(f'\n   {"Average":<30} {avg:.4f}')
print(f'\nScreenshot this -- it is your BEFORE evidence for judges.')

In [ ]:
!pip install --force-reinstall --no-deps trl>=0.15.0

## Cell 4: Run GRPO Training (200 steps, ~45 min on T4)

While this runs, you can work on Track B/C/D items.

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
)

from train.train_grpo import run_training

# Run training — adjust parameters as needed
output_dir = run_training(
    num_steps=200,           # Total training steps
    batch_size=1,            # Per-device batch size
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_generations=4,       # K=4 group size for GRPO
    max_completion_length=200,  # Max tokens per completion
    temperature=0.7,
    num_prompts=60,          # Training prompt dataset size
    save_steps=50,           # Checkpoint frequency
    logging_steps=5,
    push_to_hub=False,       # Set True to push to HF Hub
    use_wandb=False,         # Set True if WANDB_API_KEY is set
    use_unsloth=False,       # Set True if unsloth is installed
)

print(f'\nTraining complete! Model saved to: {output_dir}')

## Cell 5: Plot Reward Curve

Download this chart — embed it in the blog post and submission.

In [ ]:
from train.plot_rewards import plot_reward_curve

# Plot the training reward curve
chart_path = plot_reward_curve(
    log_file='./outputs/mini-soc-grpo/training_log.jsonl',
    output_path='./outputs/reward_curve.png',
    random_baseline=0.09,
    show_plot=True,
)

print(f'Chart saved to: {chart_path}')
print(f'Download this chart for your blog post!')

## Cell 6: Evaluate Trained Model + Before/After Comparison

Screenshot this! Before + After = your complete judge evidence.

In [ ]:
!pip install -U torchao>=0.16.0 peft>=0.15.0

In [ ]:
import os, sys
os.chdir('/content/mini-soc')
sys.path.insert(0, '/content/mini-soc')

import os, sys
os.chdir('/content/mini-soc')
sys.path.insert(0, '/content/mini-soc')

import json
import httpx
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from train.plot_rewards import plot_comparison
from peft import PeftModel

MODEL_PATH = './outputs/mini-soc-grpo'
SOC_ENV_URL = 'https://riteshp30-mini-soc.hf.space'

# Load trained model
print('Loading trained model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# Load base model first, then apply LoRA adapter
from peft import PeftModel

print('Loading trained model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map='auto',
)
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model = model.merge_and_unload()
model.eval()
print('Model loaded\n')

def run_trained_episode(task_id, max_steps=20):
    """Run one episode with the trained model."""
    result = httpx.post(
        f'{SOC_ENV_URL}/reset',
        json={'task_id': task_id},
        timeout=30
    ).json()
    total_score = 0.0

    for step in range(max_steps):
        obs_text = json.dumps(result.get('observation', {}), indent=2)[:1500]
        prompt = (
            f'You are a SOC analyst. Task: {task_id}\n'
            f'Observation:\n{obs_text}\n'
            f'Output ONE JSON action:'
        )
        inputs = tokenizer(
            prompt,
            return_tensors='pt',
            truncation=True,
            max_length=1024
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
            )

        action_text = tokenizer.decode(
            output[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )

        try:
            clean = action_text.strip()
            if clean.startswith('```'):
                lines = clean.split('\n')
                lines = [l for l in lines if not l.strip().startswith('```')]
                clean = '\n'.join(lines).strip()
            action = json.loads(clean)
        except json.JSONDecodeError:
            action = {'action_type': 'request_info', 'parameters': {}}

        result = httpx.post(
            f'{SOC_ENV_URL}/step',
            json=action,
            timeout=30
        ).json()

        if result.get('done'):
            total_score = result.get('info', {}).get('final_score', 0.0)
            break

    return total_score


# Evaluate trained model on all 3 tasks
print('Evaluating trained model...')
trained_scores = {}
for task in ['alert_triage', 'incident_investigation', 'threat_response']:
    score = run_trained_episode(task)
    trained_scores[task] = round(score, 4)
    print(f'   {task:<30} {score:.4f}')

# Print before/after comparison table
print(f'\n{"="*70}')
print(f'{"Task":<30} {"Random (Before)":>15} {"Trained (After)":>15} {"Delta":>10}')
print(f'{"-"*70}')
for task in ['alert_triage', 'incident_investigation', 'threat_response']:
    before = baseline_scores.get(task, 0.0)
    after = trained_scores.get(task, 0.0)
    delta = after - before
    sign = '+' if delta >= 0 else ''
    print(f'{task:<30} {before:>15.4f} {after:>15.4f} {sign}{delta:>9.4f}')

avg_before = sum(baseline_scores.values()) / len(baseline_scores)
avg_after = sum(trained_scores.values()) / len(trained_scores)
avg_delta = avg_after - avg_before
print(f'{"-"*70}')
print(f'{"AVERAGE":<30} {avg_before:>15.4f} {avg_after:>15.4f} {"+ " if avg_delta >= 0 else ""}{avg_delta:>8.4f}')
print(f'{"="*70}')
improvement = (avg_after / avg_before) if avg_before > 0 else float('inf')
print(f'\nImprovement: {improvement:.1f}x over random baseline')

# Generate comparison chart
chart_path = plot_comparison(
    random_scores=baseline_scores,
    trained_scores=trained_scores,
    output_path='./outputs/comparison_chart.png',
    show_plot=True,
)
print(f'\nComparison chart saved to: {chart_path}')
print(f'Screenshot this for your submission!')

## Cell 7: Push Trained Model to HuggingFace Hub (Optional)

In [ ]:
# Uncomment and run to push the trained model to HF Hub
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path='./outputs/mini-soc-grpo',
#     repo_id='riteshp30/mini-soc-grpo-v1',
#     repo_type='model',
# )
# print('Model pushed to HF Hub: riteshp30/mini-soc-grpo-v1')